# CafChem GPT — Foundation training on Colab (GPU)

This notebook clones the `SMILES_GPT` repo, picks up the foundation-model
training from the checkpoint we already built locally
(`data/GPT_ZN305_pytorch.pt`, currently 10 epochs / loss ~0.67), trains more
epochs on a Colab T4 GPU, and lets you download the updated `.pt` back to your
machine.

**Why this works:** the existing PyTorch refactor already supports resuming —
set `RESUME_FOUNDATION=1` and `run_gpt.py` loads the checkpoint and continues.
The 402 MB `ZN305K_tokenized.npz` cache is gitignored, so step 4 rebuilds it on
Colab (a few minutes, once).

Cells are meant to run top-to-bottom.

## 1. Clone the repo

The repo is public, so a plain `git clone` is all that's needed.

In [ ]:
#@title Clone SMILES_GPT
import os, shutil
REPO_DIR = "SMILES_GPT"  #@param {type:"string"}

# Wipe a stale clone so re-runs are clean.
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone https://github.com/MauricioCafiero/SMILES_GPT.git {REPO_DIR}

# Work from the repo root so all relative paths (data/, code/) resolve.
os.chdir(REPO_DIR)
print("Cwd:", os.getcwd())
print("Files:", os.listdir("."))

## 2. Install dependencies

Colab already ships `torch` (CUDA build), `pandas`, `numpy`, `Pillow`.
We only need to add `rdkit` (imported by `CafChemGPT.py`).

In [ ]:
#@title Install rdkit
# Colab already ships torch (CUDA), pandas, numpy, Pillow. Only rdkit is needed.
!pip install -q rdkit
import rdkit
print("rdkit", rdkit.__version__)

## 3. Verify the GPU and the saved checkpoint

On Colab the default device picker (`get_device`) will pick CUDA. We confirm a
T4 (or better) is attached and that the foundation checkpoint we'll resume
from is present.

In [ ]:
import torch, os
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")
else:
    print("\u26a0 No GPU — go to Runtime > Change runtime type > T4 GPU.")

ckpt = "data/GPT_ZN305_pytorch.pt"
print(f"\nFoundation checkpoint {ckpt}: "
      f"{'present' if os.path.exists(ckpt) else 'MISSING'}")
if os.path.exists(ckpt):
    print("  size:", round(os.path.getsize(ckpt)/1e6, 1), "MB")
    cfg = torch.load(ckpt, map_location="cpu", weights_only=False)
    print("  config:", cfg.get("config", cfg))

## 4. (Re)build the tokenized dataset cache

The `ZN305K_tokenized.npz` cache (402 MB) is gitignored and is **not** in the
repo, so we rebuild it once from `data/ZN305K_smiles.csv`. This takes a few
minutes.

If you'd rather not wait, you can instead upload your local `ZN305K_tokenized.npz`
to `data/` (use the Files panel on the left) and skip this cell — but rebuilding
is usually simpler than a 400 MB upload.

In [ ]:
import os, sys, numpy as np
sys.path.insert(0, "code")

from CafChemGPT import make_datasets, get_device
print("Device:", get_device())

cache = "data/ZN305K_tokenized.npz"
if os.path.exists(cache):
    z = np.load(cache)
    print(f"Cache already present: fx {z['fx'].shape}, fy {z['fy'].shape}, "
          f"VOCAB {int(z['VOCAB_SIZE'])}, max_len {int(z['max_length'])}")
else:
    print("Rebuilding tokenized cache from data/ZN305K_smiles.csv ...")
    fx, fy, VOCAB_SIZE, tokenizer, max_length = make_datasets(
        "data/ZN305K_smiles.csv", "SMILES")
    np.savez(cache, fx=fx, fy=fy, VOCAB_SIZE=VOCAB_SIZE, max_length=max_length)
    print(f"Saved {cache}: fx {fx.shape}, fy {fy.shape}, "
          f"VOCAB {VOCAB_SIZE}, max_len {max_length}")

## 5. Resume foundation training

This loads the existing checkpoint and trains `EXTRA_EPOCHS` more. On an A100
the small 2-block model runs fast (bf16 autocast is auto-enabled on CUDA), so
the default of 150 epochs is reasonable and cheap — bump it up further if loss
is still dropping at the end. Keep `batch_size=512` for dense gradient updates.

The trained model is saved back to `data/GPT_ZN305_pytorch.pt` (overwriting the
old checkpoint, so download a copy first if you want to keep the earlier
version).

In [ ]:
#@title Train more epochs on the foundation model
import os, sys, numpy as np
sys.path.insert(0, "code")

EXTRA_EPOCHS = 150  #@param {type:"integer"}
BATCH_SIZE   = 512  #@param {type:"integer"}
LR           = 1e-3 #@param {type:"number"}

from CafChemGPT import load_gpt, train_gpt, save_gpt, get_device

device = get_device()
cache = "data/ZN305K_tokenized.npz"
z = np.load(cache)
fx, fy = z["fx"], z["fy"]
VOCAB_SIZE, max_length = int(z["VOCAB_SIZE"]), int(z["max_length"])
print(f"fx {fx.shape} | fy {fy.shape} | VOCAB {VOCAB_SIZE} | max_len {max_length}")
print("Device:", device)

# train_gpt auto-enables bf16 autocast on CUDA (big speedup on an A100, no
# accuracy cost) and stays in fp32 on MPS/CPU. Keep batch_size=512 for dense
# gradient updates -- the model is small, so bigger batches buy little and
# slow per-epoch convergence.

FOUNDATION_FILE = "data/GPT_ZN305_pytorch"  # .pt appended by load/save
# total_layers=2 matches the original foundation architecture.
gpt = load_gpt(FOUNDATION_FILE, total_layers=2,
               max_length=max_length, VOCAB_SIZE=VOCAB_SIZE)

# Train EXTRA_EPOCHS more, resuming from the loaded weights.
gpt = train_gpt(gpt, fx, fy, epochs=EXTRA_EPOCHS, batch_size=BATCH_SIZE, lr=LR)

# Overwrite the checkpoint with the newly trained weights.
save_gpt(gpt, FOUNDATION_FILE)
print(f"\nFoundation checkpoint updated -> {FOUNDATION_FILE}.pt")

## 6. Download the updated `.pt` back to your Mac

Run this cell to push the checkpoint to your browser download. After it lands,
drop it into the local repo's `data/` folder (replacing the old one) and commit
if you want it versioned. The file is small (~3–4 MB).

Tip: also grab the layer-store text file next to it for completeness.

In [ ]:
from google.colab import files
import os

pt_path = "data/GPT_ZN305_pytorch.pt"
print(f"Downloading {pt_path} ({round(os.path.getsize(pt_path)/1e6,1)} MB) ...")
files.download(pt_path)

# Optional: also pull the layer-name store written by save_gpt.
ls_path = "data/layer_store_GPT_ZN305_pytorch.txt"
if os.path.exists(ls_path):
    files.download(ls_path)
    print("Also downloading", ls_path)

## 7. (Optional) Quick generation sanity check

Generate a few molecules from the freshly trained foundation to confirm it
produces valid SMILES. The more epochs trained, the fewer invalid molecules
you should see.

In [ ]:
import sys; sys.path.insert(0, "code")
from CafChemGPT import load_gpt, make_prompts, gen_mols
from smiles_tokenizer import SmilesTokenizer
from IPython.display import display
from rdkit import Chem

tok = SmilesTokenizer(vocab_file="data/vocab_305K.txt")
VOCAB = tok.vocab_size()
model = load_gpt("data/GPT_ZN305_pytorch", 2, 166, VOCAB)

prompts = make_prompts(8, 2)
pic, smiles = gen_mols(prompts, use_ramp=True, model=model, tokenizer=tok,
                       TEMP=1.5, VOCAB_SIZE=VOCAB)

# Validity rate (fraction that RDKit can parse).
valid = [s for s in smiles if Chem.MolFromSmiles(s) is not None]
print(f"{len(valid)}/{len(smiles)} valid SMILES. Samples:", smiles[:5])

# Robust display: save if possible, otherwise just display directly.
if hasattr(pic, "save"):
    pic.save("generated_molecules.png")
    from IPython.display import Image
    display(Image("generated_molecules.png"))
else:
    display(pic)